In [2]:
# Experiment 4: Effect of Embedding Dimension

# Create the corpus used for comparing different embedding dimensions
corpus_exp4 = [
    "the doctor works at the hospital",
    "the physician works at the hospital",
    "the doctor treats the patient",
    "the physician treats the patient",
    "the patient visits the hospital",
    "the doctor studies disease",
    "the physician studies disease",
    "the cat eats fish",
    "the dog eats fish",
    "the cat likes milk",
    "the dog likes meat"
]

# Convert each sentence into a list of words
sentences_exp4 = [sentence.split() for sentence in corpus_exp4]

In [3]:
from gensim.models import Word2Vec
import time

# Test different embedding dimensions
dimensions = [50, 100, 300]

models_exp4 = {}
training_times = {}

for dimension in dimensions:
    start_time = time.time()

    model_exp4 = Word2Vec(
        sentences=sentences_exp4,
        vector_size=dimension,
        window=5,
        min_count=1,
        epochs=10
    )

    end_time = time.time()

    models_exp4[dimension] = model_exp4
    training_times[dimension] = end_time - start_time

    print(f"Dimension = {dimension}")
    print(f"Training time: {training_times[dimension]:.4f} seconds")

Dimension = 50
Training time: 0.0144 seconds
Dimension = 100
Training time: 0.0111 seconds
Dimension = 300
Training time: 0.0131 seconds


In [4]:
# Compare the size of each embedding model

for dimension in dimensions:
    model = models_exp4[dimension]

    vector_count = len(model.wv)
    total_parameters = vector_count * dimension

    print(f"\nDimension = {dimension}")
    print(f"Vocabulary size: {vector_count}")
    print(f"Embedding parameters: {total_parameters}")


Dimension = 50
Vocabulary size: 18
Embedding parameters: 900

Dimension = 100
Vocabulary size: 18
Embedding parameters: 1800

Dimension = 300
Vocabulary size: 18
Embedding parameters: 5400


In [5]:
# Compare word similarity across different embedding dimensions

word_pairs = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog")
]

for dimension in dimensions:
    model = models_exp4[dimension]

    print(f"\nDimension = {dimension}")

    for word1, word2 in word_pairs:
        similarity = model.wv.similarity(word1, word2)
        print(f"{word1} - {word2}: {similarity:.4f}")


Dimension = 50
doctor - physician: 0.1657
doctor - hospital: -0.2060
cat - dog: 0.0905

Dimension = 100
doctor - physician: 0.1702
doctor - hospital: -0.0446
cat - dog: 0.0023

Dimension = 300
doctor - physician: -0.0341
doctor - hospital: -0.0349
cat - dog: 0.0494


In [6]:
# Prepare a corpus for the analogy experiment

corpus_analogy = [
    "the king is a man",
    "the queen is a woman",
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king is powerful",
    "the queen is powerful",
    "the man is strong",
    "the woman is strong",
    "the king and queen live in the palace"
]

sentences_analogy = [sentence.split() for sentence in corpus_analogy]

In [8]:
# Train a Word2Vec model for the analogy experiment

analogy_model = Word2Vec(
    sentences=sentences_analogy,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=10
)

print("Vocabulary size:", len(analogy_model.wv))
print("Vector dimension:", analogy_model.wv.vector_size)

Vocabulary size: 15
Vector dimension: 100


In [9]:
# Test the analogy: king - man + woman ≈ queen

result = analogy_model.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

print("king - man + woman:")
for word, similarity in result:
    print(f"{word}: {similarity:.4f}")

king - man + woman:
powerful: 0.1612
queen: 0.1184
is: 0.1074
palace: 0.0829
and: 0.0801


**Nhận xét**: queen đứng thứ 2, không đứng đầu. Điều này cho thấy mô hình đã học được một phần mối quan hệ giữa king, man, woman và queen, nhưng chưa thực hiện phép tương tự một cách chính xác. Corpus nhỏ và lượng dữ liệu huấn luyện hạn chế có thể là nguyên nhân khiến powerful đứng đầu.

Kết quả chưa hoàn toàn đúng kỳ vọng, nhưng vẫn cho thấy embedding đã nắm bắt được một số quan hệ giữa các từ.

In [10]:
# Prepare a simple downstream classification task

texts = [
    "the doctor treats the patient",
    "the physician works at the hospital",
    "the patient has a disease",
    "the hospital treats patients",
    "the cat eats fish",
    "the dog eats meat",
    "the cat likes milk",
    "the dog eats fish"
]

labels = [
    "medical",
    "medical",
    "medical",
    "medical",
    "animal",
    "animal",
    "animal",
    "animal"
]

print("Number of samples:", len(texts))
print("Labels:", labels)

Number of samples: 8
Labels: ['medical', 'medical', 'medical', 'medical', 'animal', 'animal', 'animal', 'animal']


In [12]:
import numpy as np

# Convert each sentence into a fixed-size vector
# by averaging the embeddings of its words

model = models_exp4[100]

sentence_vectors = []

for text in texts:
    words = text.split()

    word_vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    sentence_vector = np.mean(word_vectors, axis=0)
    sentence_vectors.append(sentence_vector)

sentence_vectors = np.array(sentence_vectors)

print("Sentence vector shape:", sentence_vectors.shape)

Sentence vector shape: (8, 100)


In [15]:
from sklearn.linear_model import LogisticRegression

# Train a simple classifier using sentence embeddings

classifier = LogisticRegression(random_state=42)

classifier.fit(sentence_vectors, labels)

predictions = classifier.predict(sentence_vectors)

print("Predictions:")
print(predictions)

Predictions:
['medical' 'medical' 'medical' 'medical' 'animal' 'animal' 'animal'
 'animal']


In [16]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(labels, predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0
